# 结果分析：复现对比表 & 净值曲线

本 Notebook 演示：
1. 从 `runs/` 目录汇总多个实验结果
2. 生成对比表格（对应 README 中的结果表）
3. 并排绘制净值曲线

In [ ]:
import sys, os, json, glob
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. 汇总实验结果

In [ ]:
RUNS_DIR = '../runs'

rows = []
for path in sorted(glob.glob(f'{RUNS_DIR}/*/results.json')):
    exp_name = os.path.basename(os.path.dirname(path))
    with open(path) as f:
        data = json.load(f)
    for split in ('val', 'test'):
        r = data.get(split, {})
        if not r:
            continue
        rows.append({
            'experiment': exp_name,
            'split': split,
            'ann_return':  f"{r.get('annual_return', 0):.2%}",
            'sharpe':      f"{r.get('sharpe_ratio', 0):.3f}",
            'IR':          f"{r.get('information_ratio', 0):.3f}",
            'MDD':         f"{r.get('max_drawdown', 0):.2%}",
            'turnover':    f"{r.get('avg_turnover_rate', 0):.2%}",
            'bench_ret':   f"{r.get('benchmark_total_return', 0):.2%}",
        })

if rows:
    df = pd.DataFrame(rows)
    display(df)
    df.to_csv('../results/summary_table.csv', index=False)
    print('已保存到 results/summary_table.csv')
else:
    print('未找到实验结果。请先运行 scripts/train.py 完成训练。')

## 2. 净值曲线对比（多实验）

In [ ]:
# 从各实验目录读取 portfolio_value（如果你在 train.py 中额外保存了的话）
# 此处演示用随机模拟数据
import numpy as np
np.random.seed(42)
T = 200

def simulate_nav(mu=0.0003, sigma=0.015, T=200):
    returns = np.random.normal(mu, sigma, T)
    return np.cumprod(1 + returns)

fig, ax = plt.subplots(figsize=(12, 5))

nav_main     = simulate_nav(mu=0.0005, sigma=0.012)
nav_vanilla  = simulate_nav(mu=0.0002, sigma=0.015)
nav_ablation = simulate_nav(mu=0.0003, sigma=0.013)
nav_bench    = simulate_nav(mu=0.0001, sigma=0.010)

ax.plot(nav_main,     label='iTransformer + MORL + Curriculum', linewidth=2.0)
ax.plot(nav_vanilla,  label='Vanilla Baseline',                  linewidth=1.5, linestyle='--')
ax.plot(nav_ablation, label='消融：无课程学习',                    linewidth=1.5, linestyle='-.')
ax.plot(nav_bench,    label='等权基准',                           linewidth=1.5, color='gray')

ax.set_title('各实验净值曲线对比（测试集）\n注：此为示例数据，替换为真实回测结果后使用', fontsize=13)
ax.set_xlabel('交易日')
ax.set_ylabel('净值')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/nav_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. 单实验深度分析

In [ ]:
# 读取单个实验结果做详细分析
result_path = '../runs/itransformer_morl/results.json'

if os.path.exists(result_path):
    with open(result_path) as f:
        res = json.load(f)
    test = res.get('test', {})
    
    metrics = {
        '年化收益率':   f"{test.get('annual_return', 0):.2%}",
        '夏普比率':     f"{test.get('sharpe_ratio', 0):.3f}",
        '信息比率':     f"{test.get('information_ratio', 0):.3f}",
        '最大回撤':     f"{test.get('max_drawdown', 0):.2%}",
        '基准年化收益': f"{test.get('benchmark_annual_return', 0):.2%}",
        '年化超额收益': f"{test.get('excess_return', 0):.2%}",
        '日均换手率':   f"{test.get('avg_turnover_rate', 0):.2%}",
        '平均持仓数':   f"{test.get('avg_holdings', 0):.1f}",
        '成本率':       f"{test.get('transaction_cost_ratio', 0):.2%}",
    }
    display(pd.DataFrame.from_dict(metrics, orient='index', columns=['值']))
else:
    print(f'未找到结果文件: {result_path}\n请先运行训练脚本。')